In [14]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
%load_ext kedro.ipython
%reload_kedro --params dataset_name=diabetes

In [15]:
from project.example_addition import addition

print(addition(5, 5))

10


In [16]:
from matplotlib import pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.preprocessing import StandardScaler

from kedro.io import DataCatalog
from kedro.config import OmegaConfigLoader
from omegaconf.resolvers import oc # TODO: В ИНТСРУКЦИЮ 

conf_loader = OmegaConfigLoader(
    conf_source="../conf/base",
    custom_resolvers={"oc.env": oc.env}
    )
catalog_config = conf_loader.get("catalog")

catalog = DataCatalog.from_config(catalog_config)

df = catalog.load("source@raw")

/Users/sergejsvetsov/Git/python-project-template/.venv/lib/python3.12/site-packages/kedro/config/omegaconf_config.py:701: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(name=name, resolver=resolver)
/Users/sergejsvetsov/Git/python-project-template/.venv/lib/python3.12/site-packages/kedro/config/omegaconf_config.py:637: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_resolver() instead.
See https://github.com/hydra-ecosystem/omegaconf/issues/426 for migration instructions.

  OmegaConf.register_new_resolver(
/Users/sergejsvetsov/Git/python-project-template/.venv/lib/python3.12/site-packages/kedro/config/omegaconf_config.py:644: UserWarning: register_new_resolver() is deprecated and will be removed in a future release.
Use register_re

In [17]:
from project.modeling.train import train_model, create_model, create_scaler
from project.modeling.split_data import split_data
from project.modeling.validation import cross_validate_model

X_train, X_test, y_train, y_test = split_data(df, target_col="MEDV")

model = create_model(scaler="standard", alpha=0.1, l1_ratio=0.5, max_iter=10000)


cv_rmse = cross_validate_model(
    model=model,
    X_train=X_train,
    y_train=y_train,
)
print(f"CV MAE: {cv_rmse:.3f}")

CV MAE: 3.371


In [18]:
from project.modeling.tuning import tune_model

search_space = {
    "alpha": {
        "low": 0.0001,
        "high": 10.0
    },
    "l1_ratio": {
        "low": 0.0,
        "high": 1.0
    }
}

study = tune_model(
    X_train=X_train,
    y_train=y_train,
    search_space=search_space,
    scaler="standard",
    n_trials=30,
    n_splits=5,
)

print("Лучшие параметры:", study.best_params)
print("Лучший CV RMSE:", study.best_value)

[I 2026-11-05 20:37:08,780] A new study created in memory with name: no-name-de5efa1e-fac1-4169-95fe-c090df562e6f
[I 2026-11-05 20:37:08,789] Trial 0 finished with value: 3.4206354952366977 and parameters: {'alpha': 0.0074593432857265485, 'l1_ratio': 0.9507143064099162}. Best is trial 0 with value: 3.4206354952366977.
[I 2026-11-05 20:37:08,797] Trial 1 finished with value: 3.5316769037249416 and parameters: {'alpha': 0.45705630998014507, 'l1_ratio': 0.5986584841970366}. Best is trial 0 with value: 3.4206354952366977.
[I 2026-11-05 20:37:08,805] Trial 2 finished with value: 3.4270153550941993 and parameters: {'alpha': 0.0006026889128682511, 'l1_ratio': 0.15599452033620265}. Best is trial 0 with value: 3.4206354952366977.
[I 2026-11-05 20:37:08,813] Trial 3 finished with value: 3.42815435228052 and parameters: {'alpha': 0.00019517224641449495, 'l1_ratio': 0.8661761457749352}. Best is trial 0 with value: 3.4206354952366977.
[I 2026-11-05 20:37:08,821] Trial 4 finished with value: 3.37447

Лучшие параметры: {'alpha': 0.08775646984512295, 'l1_ratio': 0.1791123637589616}
Лучший CV RMSE: 3.3655101758014276


In [19]:
best_model = train_model(
    X_train=X_train,
    y_train=y_train,
    scaler="standard",
    max_iter=10_000,
    **study.best_params,
)

In [20]:
from project.modeling.evaluate import evaluate_model

test_metrics = evaluate_model(
    model=best_model,
    X_test=X_test,
    y_test=y_test,
)

print(test_metrics)

{'mae': {'value': 3.1660739477721167, 'step': 0}, 'rmse': {'value': 4.992062070695953, 'step': 0}, 'r2': {'value': 0.6601745792495206, 'step': 0}}


In [21]:
import kedro
import kedro_mlflow


In [22]:
%reload_kedro --params dataset_name=diabetes

UsageError: Line magic function `%reload_kedro` not found.


In [ ]:
import mlflow
from sklearn.dummy import DummyRegressor

from project.modeling.evaluate import evaluate_model
from project.modeling.split_data import split_data

params = catalog.load("parameters")

X_train, X_test, y_train, y_test = split_data(
    df_dataset=catalog.load("source@raw"),
    target_col=params["target_col"],
    **params["split_options"],
)

model = DummyRegressor(strategy="median")
model.fit(X_train, y_train)

metrics = evaluate_model(model=model, X_test=X_test, y_test=y_test)

mlflow.set_tracking_uri("http://127.0.0.1:5001")
mlflow.set_experiment("regression")  # Тот же эксперимент, что у ElasticNet.

with mlflow.start_run(run_name="diabetes-median"):
    mlflow.log_param("dataset_name", "diabetes")
    mlflow.set_tag("dataset", "diabetes")
    mlflow.log_param("strategy", "median")
    mlflow.log_metrics({name: record["value"] for name, record in metrics.items()})

/Users/sergejsvetsov/Git/python-project-template/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'catalog' is not defined